# 🐍 Práctica 13 - Sprint 4 - Recursión

**Tema:** recursión

> Cada ejercicio incluye su enunciado, los datos de partida y la solución; la salida de cada celda es la que produce el código.

# Sección 7. Recursión

## Ejercicio 7.1 - Profundidad de un documento

Algunos sistemas de destino limitan el número de niveles de anidamiento que aceptan. Escribe `profundidad(nodo)` con estas reglas: un valor simple tiene profundidad 0, y un diccionario o una lista tienen profundidad 1 más la mayor profundidad de sus elementos. Un diccionario o una lista vacíos tienen profundidad 1.

Calcula la profundidad de `documento`, de `documento["items"]` y del valor `42`.

**Salida esperada**

```text
Profundidad: 4
Profundidad de 'items': 3
Profundidad de un valor simple: 0
```

In [1]:
documento = {
    "id": 1,
    "cliente": {"nombre": "Acme", "contacto": {"email": "a@acme.com", "tel": {"pref": "+34"}}},
    "items": [{"sku": "A1", "atributos": {"color": "rojo"}}, {"sku": "B2"}],
}

In [2]:
def profundidad(nodo):
    if isinstance(nodo, dict):
        hijos = nodo.values()
    elif isinstance(nodo, list):
        hijos = nodo
    else:
        return 0
    return 1 + max((profundidad(h) for h in hijos), default=0)


print("Profundidad:", profundidad(documento))
print("Profundidad de 'items':", profundidad(documento["items"]))
print("Profundidad de un valor simple:", profundidad(42))

Profundidad: 4
Profundidad de 'items': 3
Profundidad de un valor simple: 0


## Ejercicio 7.2 - Buscar ficheros por extensión



Escribe `buscar_ficheros(nodo, extension, ruta="")`, que devuelva la lista de rutas completas de los ficheros con esa extensión. Usa `nombre.endswith(extension)`. Muestra las rutas `.parquet` y su total, y después solo el total de `.csv`.

**Salida esperada**

```text
/plata/ventas/year=2026/part-0000.parquet
/plata/ventas/year=2026/part-0001.parquet
/plata/clientes/part-0000.parquet
/oro/kpis_mensuales.parquet
Total .parquet: 4
Total .csv: 3
```

In [3]:
lake = {
    "bronce": {"ventas": {"2026-09.csv": 120, "2026-10.csv": 135}, "clientes.csv": 40},
    "plata": {
        "ventas": {"year=2026": {"part-0000.parquet": 30, "part-0001.parquet": 28}},
        "clientes": {"part-0000.parquet": 12},
    },
    "oro": {"kpis_mensuales.parquet": 5, "README.md": 1},
}

In [4]:
def buscar_ficheros(nodo, extension, ruta=""):
    encontrados = []
    for nombre, hijo in nodo.items():
        ruta_hijo = f"{ruta}/{nombre}"
        if isinstance(hijo, dict):
            encontrados.extend(buscar_ficheros(hijo, extension, ruta_hijo))
        elif nombre.endswith(extension):
            encontrados.append(ruta_hijo)
    return encontrados


parquets = buscar_ficheros(lake, ".parquet")
for ruta in parquets:
    print(ruta)
print("Total .parquet:", len(parquets))
print("Total .csv:", len(buscar_ficheros(lake, ".csv")))

/plata/ventas/year=2026/part-0000.parquet
/plata/ventas/year=2026/part-0001.parquet
/plata/clientes/part-0000.parquet
/oro/kpis_mensuales.parquet
Total .parquet: 4
Total .csv: 3


## Ejercicio 7.3 - Linaje de tablas (ancestros)

Este diccionario indica las tablas de origen directas de cada tabla.

Escribe `ancestros(tabla)`, que devuelva un `set` con todas las tablas de las que depende, directa o indirectamente. Muestra los ancestros de `dashboard_ventas`, de `ventas_limpias` y de `ventas_raw`. Por último, con una comprehension, obtén las fuentes originales del dashboard: los ancestros que no tienen ningún origen.

**Salida esperada**

```text
['clientes_limpios', 'clientes_raw', 'dim_tiempo', 'kpis_ventas', 'tipos_cambio', 'ventas_limpias', 'ventas_raw']
['tipos_cambio', 'ventas_raw']
[]
Fuentes originales del dashboard: ['clientes_raw', 'dim_tiempo', 'tipos_cambio', 'ventas_raw']
```

In [5]:
origenes = {
    "dashboard_ventas": ["kpis_ventas", "dim_tiempo"],
    "kpis_ventas": ["ventas_limpias", "clientes_limpios"],
    "ventas_limpias": ["ventas_raw", "tipos_cambio"],
    "clientes_limpios": ["clientes_raw"],
    "dim_tiempo": [],
    "ventas_raw": [],
    "clientes_raw": [],
    "tipos_cambio": [],
}

In [6]:
def ancestros(tabla):
    resultado = set()
    for origen in origenes[tabla]:
        resultado.add(origen)
        resultado |= ancestros(origen)
    return resultado


del_dashboard = ancestros("dashboard_ventas")
print(sorted(del_dashboard))
print(sorted(ancestros("ventas_limpias")))
print(sorted(ancestros("ventas_raw")))
print("Fuentes originales del dashboard:", sorted([t for t in del_dashboard if not origenes[t]]))

['clientes_limpios', 'clientes_raw', 'dim_tiempo', 'kpis_ventas', 'tipos_cambio', 'ventas_limpias', 'ventas_raw']
['tipos_cambio', 'ventas_raw']
[]
Fuentes originales del dashboard: ['clientes_raw', 'dim_tiempo', 'tipos_cambio', 'ventas_raw']


## Ejercicio 7.4 - Contar nulos en una respuesta de API

La respuesta mezcla diccionarios y listas a varios niveles.

Escribe `contar_nulos(nodo)`, que cuente todos los `None` de la estructura, estén dentro de diccionarios o de listas.

**Salida esperada**

```text
Nulos en la respuesta: 6
Nulos en 'lineas': 3
```

In [7]:
respuesta_api = {
    "pedido": 9001,
    "cliente": {"nombre": "Acme", "email": None, "telefonos": ["600111222", None]},
    "lineas": [
        {"sku": "A1", "precio": 10.0, "descuento": None},
        {"sku": None, "precio": None, "descuento": 0.1},
    ],
    "notas": None,
}

In [8]:
def contar_nulos(nodo):
    if nodo is None:
        return 1
    if isinstance(nodo, dict):
        return sum(contar_nulos(v) for v in nodo.values())
    if isinstance(nodo, list):
        return sum(contar_nulos(v) for v in nodo)
    return 0


print("Nulos en la respuesta:", contar_nulos(respuesta_api))
print("Nulos en 'lineas':", contar_nulos(respuesta_api["lineas"]))

Nulos en la respuesta: 6
Nulos en 'lineas': 3


## Ejercicio 7.5 - Detectar ciclos en un DAG

Un orquestador rechaza un flujo de tareas si sus dependencias forman un ciclo (por ejemplo, `b → c → d → b`), porque ninguna de esas tareas podría arrancar nunca.

1. Escribe `tiene_ciclo(grafo, nodo, camino)`, que devuelva `True` si, partiendo de `nodo`, se vuelve a un nodo que ya está en el camino actual. `camino` es un `set` con los nodos visitados en la rama que se está recorriendo.
2. Escribe `validar_dag(grafo)`, que compruebe todos los nodos y devuelva `True` si no hay ningún ciclo.

**Salida esperada**

```text
dag_ok es un DAG válido: True
dag_ko es un DAG válido: False
```

In [9]:
dag_ok = {"a": ["b", "c"], "b": ["d"], "c": ["d"], "d": []}
dag_ko = {"a": ["b"], "b": ["c"], "c": ["d"], "d": ["b"]}

In [10]:
def tiene_ciclo(grafo, nodo, camino):
    if nodo in camino:
        return True
    camino = camino | {nodo}
    for vecino in grafo[nodo]:
        if tiene_ciclo(grafo, vecino, camino):
            return True
    return False


def validar_dag(grafo):
    return not any(tiene_ciclo(grafo, nodo, set()) for nodo in grafo)


print("dag_ok es un DAG válido:", validar_dag(dag_ok))
print("dag_ko es un DAG válido:", validar_dag(dag_ko))

dag_ok es un DAG válido: True
dag_ko es un DAG válido: False


## Ejercicio 7.6 - Tamaños legibles

Los sistemas de almacenamiento informan del tamaño en bytes, pero en los informes se muestran en la unidad más adecuada: KB, MB, GB…

Escribe `legible(valor, nivel=0)` de forma **recursiva**, usando la lista de unidades `["B", "KB", "MB", "GB", "TB", "PB"]`:

- **Caso base:** si el valor es menor que 1024, o ya no quedan unidades mayores, se devuelve con un decimal y su unidad.
- **Caso recursivo:** se divide el valor entre 1024 y se llama con el nivel siguiente.

**Salida esperada**

```text
             512 -> 512.0 B
            2048 -> 2.0 KB
      5368709120 -> 5.0 GB
   1099511627776 -> 1.0 TB
       734003200 -> 700.0 MB
```

In [11]:
tamanos_bytes = [512, 2048, 5_368_709_120, 1_099_511_627_776, 734_003_200]

In [12]:
UNIDADES = ["B", "KB", "MB", "GB", "TB", "PB"]


def legible(valor, nivel=0):
    if valor < 1024 or nivel == len(UNIDADES) - 1:
        return f"{valor:.1f} {UNIDADES[nivel]}"
    return legible(valor / 1024, nivel + 1)


for t in tamanos_bytes:
    print(f"{t:>16} -> {legible(t)}")

             512 -> 512.0 B
            2048 -> 2.0 KB
      5368709120 -> 5.0 GB
   1099511627776 -> 1.0 TB
       734003200 -> 700.0 MB


## Ejercicio 7.7 - `tree_reduce`

Cuando hay muchas particiones, combinar todos sus resultados en un solo nodo puede saturarlo. Spark ofrece `treeReduce()`, que combina los resultados parciales por parejas, en varios niveles, como un torneo.

1. Escribe `combinar(a, b)`, que devuelva un diccionario nuevo con la suma de los conteos de los dos diccionarios. No es recursiva.
2. Escribe `tree_reduce(parciales, nivel=0)`, que divida la lista en dos mitades, se llame con cada una y combine los dos resultados. El caso base es una lista con un solo diccionario. Imprime en cada llamada el nivel y el número de parciales que recibe, con una sangría de dos espacios por nivel.
3. Muestra el resultado final con las claves ordenadas.

**Salida esperada**

```text
nivel 0: 6 parciales
  nivel 1: 3 parciales
    nivel 2: 1 parciales
    nivel 2: 2 parciales
      nivel 3: 1 parciales
      nivel 3: 1 parciales
  nivel 1: 3 parciales
    nivel 2: 1 parciales
    nivel 2: 2 parciales
      nivel 3: 1 parciales
      nivel 3: 1 parciales
Resultado: {'click': 408, 'compra': 23, 'view': 1085}
```

In [13]:
parciales = [
    {"click": 120, "view": 400},
    {"click": 98, "compra": 7},
    {"view": 310, "compra": 3},
    {"click": 150, "view": 280, "compra": 12},
    {"view": 95},
    {"click": 40, "compra": 1},
]

In [14]:
def combinar(a, b):
    """Suma los conteos de dos diccionarios en uno nuevo."""
    resultado = dict(a)
    for clave, valor in b.items():
        resultado[clave] = resultado.get(clave, 0) + valor
    return resultado


def tree_reduce(parciales, nivel=0):
    print("  " * nivel + f"nivel {nivel}: {len(parciales)} parciales")
    if len(parciales) == 1:
        return parciales[0]
    mitad = len(parciales) // 2
    izquierda = tree_reduce(parciales[:mitad], nivel + 1)
    derecha = tree_reduce(parciales[mitad:], nivel + 1)
    return combinar(izquierda, derecha)


resultado = tree_reduce(parciales)
print("Resultado:", dict(sorted(resultado.items())))

nivel 0: 6 parciales
  nivel 1: 3 parciales
    nivel 2: 1 parciales
    nivel 2: 2 parciales
      nivel 3: 1 parciales
      nivel 3: 1 parciales
  nivel 1: 3 parciales
    nivel 2: 1 parciales
    nivel 2: 2 parciales
      nivel 3: 1 parciales
      nivel 3: 1 parciales
Resultado: {'click': 408, 'compra': 23, 'view': 1085}


## Ejercicio 7.8 - Detectar carpetas con ficheros pequeños

En HDFS y en los data lakes, miles de ficheros diminutos degradan el rendimiento: cada fichero supone metadatos y una tarea de lectura. Se recomienda compactar las carpetas donde predominan los ficheros pequeños.

1. Escribe `ficheros_pequenos(nodo, umbral)`, que recorra una carpeta y todas sus subcarpetas y devuelva una tupla `(total_de_ficheros, ficheros_por_debajo_del_umbral)`.
2. Aplícala a cada partición de fecha y marca `COMPACTAR` las que tengan más de la mitad de sus ficheros por debajo del umbral.
3. Aplícala también a la tabla completa.

**Salida esperada**

```text
fecha=2026-10-13: 2 ficheros, 0 pequeños -> ok
fecha=2026-10-14: 6 ficheros, 6 pequeños -> COMPACTAR
fecha=2026-10-15: 3 ficheros, 2 pequeños -> COMPACTAR
Tabla completa (ficheros, pequeños): (11, 8)
```

In [15]:
tabla_eventos = {
    "fecha=2026-10-13": {"part-0.parquet": 210, "part-1.parquet": 190},
    "fecha=2026-10-14": {f"part-{i}.parquet": 3 for i in range(6)},
    "fecha=2026-10-15": {
        "hora=00": {"part-0.parquet": 2, "part-1.parquet": 4},
        "hora=01": {"part-0.parquet": 150},
    },
}
umbral_mb = 32

In [16]:
def ficheros_pequenos(nodo, umbral):
    """Devuelve (total_de_ficheros, ficheros_por_debajo_del_umbral)."""
    total = pequenos = 0
    for hijo in nodo.values():
        if isinstance(hijo, dict):
            t, p = ficheros_pequenos(hijo, umbral)
            total += t
            pequenos += p
        else:
            total += 1
            if hijo < umbral:
                pequenos += 1
    return total, pequenos


for particion, contenido in tabla_eventos.items():
    total, pequenos = ficheros_pequenos(contenido, umbral_mb)
    marca = "COMPACTAR" if pequenos > total / 2 else "ok"
    print(f"{particion}: {total} ficheros, {pequenos} pequeños -> {marca}")

print("Tabla completa (ficheros, pequeños):", ficheros_pequenos(tabla_eventos, umbral_mb))

fecha=2026-10-13: 2 ficheros, 0 pequeños -> ok
fecha=2026-10-14: 6 ficheros, 6 pequeños -> COMPACTAR
fecha=2026-10-15: 3 ficheros, 2 pequeños -> COMPACTAR
Tabla completa (ficheros, pequeños): (11, 8)


## Ejercicio 7.9 - Trazas de un agente de IA

Las plataformas de observabilidad de agentes guardan cada ejecución como un árbol de pasos: llamadas al LLM, llamadas a herramientas y subagentes, que a su vez lanzan otros pasos.

Escribe cuatro funciones recursivas:

- `tokens_totales(nodo)`: suma los tokens del nodo y de todos sus descendientes.
- `profundidad(nodo)`: un paso sin hijos tiene profundidad 1.
- `herramientas(nodo)`: devuelve el `set` de herramientas usadas en los pasos de tipo `tool`.
- `llamadas_llm(nodo)`: cuenta los pasos de tipo `llm`.

Muestra los resultados para la traza completa y los tokens consumidos solo por `subagente_analista`.

**Salida esperada**

```text
Tokens totales: 6050
Profundidad máxima: 4
Herramientas usadas: ['python', 'rag', 'sql']
Llamadas al LLM: 4
Tokens del subagente: 3050
```

In [17]:
traza = {
    "paso": "agente_principal", "tipo": "llm", "tokens": 1200,
    "hijos": [
        {"paso": "buscar_clientes", "tipo": "tool", "herramienta": "sql", "tokens": 0, "hijos": []},
        {"paso": "subagente_analista", "tipo": "llm", "tokens": 2300, "hijos": [
            {"paso": "calcular_churn", "tipo": "tool", "herramienta": "python", "tokens": 0, "hijos": []},
            {"paso": "buscar_contexto", "tipo": "tool", "herramienta": "rag", "tokens": 450, "hijos": [
                {"paso": "reescribir_consulta", "tipo": "llm", "tokens": 300, "hijos": []},
            ]},
        ]},
        {"paso": "redactar_respuesta", "tipo": "llm", "tokens": 1800, "hijos": []},
    ],
}

In [18]:
def tokens_totales(nodo):
    return nodo["tokens"] + sum(tokens_totales(h) for h in nodo["hijos"])


def profundidad(nodo):
    return 1 + max((profundidad(h) for h in nodo["hijos"]), default=0)


def herramientas(nodo):
    usadas = {nodo["herramienta"]} if nodo["tipo"] == "tool" else set()
    for h in nodo["hijos"]:
        usadas |= herramientas(h)
    return usadas


def llamadas_llm(nodo):
    return (1 if nodo["tipo"] == "llm" else 0) + sum(llamadas_llm(h) for h in nodo["hijos"])


subagente = traza["hijos"][1]
print("Tokens totales:", tokens_totales(traza))
print("Profundidad máxima:", profundidad(traza))
print("Herramientas usadas:", sorted(herramientas(traza)))
print("Llamadas al LLM:", llamadas_llm(traza))
print("Tokens del subagente:", tokens_totales(subagente))

Tokens totales: 6050
Profundidad máxima: 4
Herramientas usadas: ['python', 'rag', 'sql']
Llamadas al LLM: 4
Tokens del subagente: 3050


## Ejercicio 7.10 - Resumen jerárquico

Cuando un documento no cabe en el contexto de un LLM, se usa un resumen jerárquico: se resumen los fragmentos por grupos, después se resumen esos resúmenes, y así hasta obtener uno solo. Como aquí no llamamos a un modelo real, la función `resumir()` lo simula quedándose con las 3 primeras palabras de cada texto.

1. Escribe `resumen_jerarquico(textos, grupo=2, nivel=0)`. En cada llamada imprime el nivel y cuántos textos recibe. Si solo queda un texto, lo devuelve. Si no, resume los textos en grupos de `grupo` elementos y se llama con la lista de resúmenes.
2. Ejecútala con grupos de 2 y de 3 y compara cuántos niveles necesita cada una.
3. Observa el resumen final: ¿qué información de los capítulos se ha perdido? Con un LLM real ocurre algo parecido, por eso el tamaño de grupo y las instrucciones de cada nivel influyen en la calidad del resultado.

**Salida esperada**

```text
Nivel 0: 5 textos
Nivel 1: 3 textos
Nivel 2: 2 textos
Nivel 3: 1 textos
Resumen final: Ventas crecen un Nuevo almacén en

Nivel 0: 5 textos
Nivel 1: 2 textos
Nivel 2: 1 textos
Resumen final: Ventas crecen un Coste logístico sube
```

In [19]:
def resumir(textos):
    return " ".join([" ".join(t.split()[:3]) for t in textos])

capitulos = [
    "Ventas crecen un 12% en el canal online",
    "Tiendas físicas estables respecto al trimestre anterior",
    "Devoluciones bajan tras el cambio de política",
    "Coste logístico sube por el precio del combustible",
    "Nuevo almacén en Valencia reduce plazos de entrega",
]

In [20]:
def resumen_jerarquico(textos, grupo=2, nivel=0):
    print(f"Nivel {nivel}: {len(textos)} textos")
    if len(textos) == 1:
        return textos[0]
    resumenes = [resumir(textos[i:i + grupo]) for i in range(0, len(textos), grupo)]
    return resumen_jerarquico(resumenes, grupo, nivel + 1)


print("Resumen final:", resumen_jerarquico(capitulos, grupo=2))
print()
print("Resumen final:", resumen_jerarquico(capitulos, grupo=3))

Nivel 0: 5 textos
Nivel 1: 3 textos
Nivel 2: 2 textos
Nivel 3: 1 textos
Resumen final: Ventas crecen un Nuevo almacén en

Nivel 0: 5 textos
Nivel 1: 2 textos
Nivel 2: 1 textos
Resumen final: Ventas crecen un Coste logístico sube


**2. Comparación**: con grupos de 2 hacen falta 4 niveles (5 → 3 → 2 → 1); con grupos de 3, solo 3 niveles (5 → 2 → 1). Cuanto mayor es el grupo, menos niveles.

**3. Información perdida**: el resumen final solo conserva fragmentos de dos o tres capítulos (por ejemplo, `Ventas crecen un` y `Nuevo almacén en`) y se pierde todo lo relacionado con tiendas físicas, devoluciones y costes logísticos. Cada nivel descarta información, y lo que se descarta en un nivel ya no se puede recuperar en el siguiente. Con un LLM real pasa algo parecido: el tamaño del grupo y las instrucciones de cada nivel (qué conservar, qué longitud) determinan la calidad del resumen final.